In [1]:
# Import the basic spark library
from pyspark.sql import SparkSession

# Create an entry point to the PySpark Application
spark = SparkSession.builder \
      .master("local") \
      .appName("ProjSMBUD") \
      .getOrCreate()
# master contains the URL of your remote spark instance or 'local'

In [2]:
# Load DataFrames (we prefer directly them to the rdd, because we already had csv with headers)
#Configuring them with some examples to check the correctness (now commented before and after the casting)

df_incollection = spark.read.option("header", True).option("delimiter", ";")\
                .csv("incollectionFormatted.csv.txt")
df_book = spark.read.option("header", True).option("delimiter", ";")\
                .csv("bookFormatted.csv.txt")
df_author = spark.read.option("header", True).option("delimiter", ",")\
                .csv("authorFormatted.csv.txt")

#Load bridge tables? (No, now they are in the entities), #they had already id string types, but now we don't have them





# We use of datatypes from pyspark.sql.types
from pyspark.sql.types import StringType, DateType, FloatType, StructType, StructField, ArrayType
from pyspark.sql.functions import * 


##We use the ids in string format!
##TO HAVE THE ARRAY OF STRINGS
df_book = df_book\
    .select("id","date", "url", "isbn", "citationscore", "publisher" , "title", split(col("editor_array"),",").alias("editor_array"))
df_incollection = df_incollection\
    .select("id","title","date", "url", "pages", "citationscore", \
    "book_id", split(col("author_array"),",").alias("author_array"), \
    split(col("cited_inc_array"),",").alias("cited_inc_array"))

#df_incollection.printSchema()
#df_incollection.limit(5).select("id", 'author_array').show()
df_incollection = df_incollection \
  .withColumn("id" ,
              df_incollection["id"]
              .cast(StringType()))   \
  .withColumn("title"  ,
              df_incollection["title"]
              .cast(StringType())) \
  .withColumn("date",
              df_incollection["date"]
              .cast(DateType()))    \
  .withColumn("url"  ,
              df_incollection["url"]
              .cast(StringType())) \
  .withColumn("pages"  ,
              df_incollection["pages"]
              .cast(StringType())) \
  .withColumn("citationscore"  ,
              df_incollection["citationscore"]
              .cast(FloatType())) \
#  .withColumn("author_array"  ,
#              df_incollection["author_array"]
#              .cast(ArrayType(StringType()))) \
#  .withColumn("cited_inc_array"  ,
#              df_incollection["cited_inc_array"]
#              .cast(ArrayType(StringType()))) \
#df_incollection.printSchema()
#df_incollection.filter(array_contains(df_incollection.author_array,"7370")).select("id","author_array").show(truncate = False)
#5371|[7371,  7370,  7372]
#df_incollection.select(explode(df_incollection.cited_inc_array)).show()



#df_book.printSchema()
#df_book.show()
df_book = df_book \
  .withColumn("id" ,
              df_book["id"]
              .cast(StringType()))   \
  .withColumn("isbn"  ,
              df_book["isbn"]
              .cast(StringType())) \
  .withColumn("date",
              df_book["date"]
              .cast(DateType()))    \
  .withColumn("url"  ,
              df_book["url"]
              .cast(StringType())) \
  .withColumn("publisher"  ,
              df_book["publisher"]
              .cast(StringType())) \
  .withColumn("citationscore"  ,
              df_book["citationscore"]
              .cast(FloatType())) \
  .withColumn("title"  ,
              df_book["title"]
              .cast(StringType())) \
#  .withColumn("editors_array"  ,
#              df_book_editors["editors_array"]) \
#df_book.printSchema()
#df_book.select(explode(df_book.editor_array)).show()


#df_author.printSchema()
#df_author.limit(5).show()
df_author = df_author \
  .withColumn("id" ,
              df_author["id"]
              .cast(StringType()))   \
  .withColumn("name"  ,
              df_author["name"]
              .cast(StringType())) \
  .withColumn("citationscore",
              df_author["citationscore"]
              .cast(FloatType()))    \
#email is already a string
#df_author.printSchema() 
#df_author.limit(5).show()

In [3]:
#WHERE-JOIN
#df_author - df_book - df_incollection
# This query returns name and pages of incollections contained in the book with the highest citationscore and an existing url as attribute.
# All the incollections shown in the results need to have an existing number of pages as attribute.

df_book   \
            .filter(col("url") != "null")    \
            .select("id", "citationscore")    \
            .filter(col("citationscore") == df_book.groupby().max("citationscore")   \
                .first().asDict()['max(citationscore)'])    \
            .select("id")    \
            .join(df_incollection.select("id", "title", "pages", "book_id"),    \
                  (df_book.id).isin(df_incollection.book_id))    \
            .select("title", "pages")   \
            .filter(col("pages") != "null")    \
            .withColumnRenamed("title","incollection_name")    \
            .show(truncate = False)

+-----------------------------------------------------------+---------+
|incollection_name                                          |pages    |
+-----------------------------------------------------------+---------+
|Near Field Communication.                                  |829      |
|Biometric Recognition.                                     |104      |
|FPGAs in Cryptography.                                     |499-501  |
|XML-Based Access Control Languages.                        |1395-1399|
|Fermats Little Theorem.                                    |456      |
|Chosen Plaintext Attack.                                   |205-206  |
|Worms in Cellular Networks.                                |1392-1393|
|Chip Card.                                                 |205      |
|EMV.                                                       |412-416  |
|Hash Agility.                                              |538-540  |
|Biba Integrity Model.                                      |81 

In [4]:
#GROUP BY, HAVING, AS
# This query returns count of incollections having more than 5 citated incollection x year.

df_incollection   \
        .filter(size(col("cited_inc_array")) > 5)    \
        .select(year("date"))    \
        .withColumnRenamed("year(date)","year")    \
        .groupby("year")    \
        .count()    \
        .withColumnRenamed("count","num_incollections")    \
        .show(truncate = False)

+----+-----------------+
|year|num_incollections|
+----+-----------------+
|2018|25               |
|2022|17               |
|2013|1                |
|2014|1                |
|2019|28               |
|2020|15               |
|2012|1                |
|2016|3                |
|2011|86               |
|2017|1732             |
|2021|13               |
+----+-----------------+



In [74]:
#Creation
# This command creates add a book to the df_book, results are saved in df_final. 
# Some queries are made to show that tha book has been inserted successfully with the biggest id.

from datetime import datetime

columns = StructType([
    StructField('id', StringType(), False),
    StructField('date', DateType(), False),
    StructField('url', StringType(), False),
    StructField('isbn', StringType(), False),
    StructField('citationscore', FloatType(), False),
    StructField('publisher', StringType(), False),
    StructField('title', StringType(), False),
    StructField('editor_array', ArrayType(StringType()), False)
])

add_row = [['5180', datetime.strptime('2022-10-02', '%Y-%m-%d'), 'https://www.theblackhole.com', '978-8-817-08378-2', 0.5652352, 'Politecnico di Milano', 'The black hole', [5052, 5180]]]
add_row_df = spark.createDataFrame(add_row, columns)

df_final = df_book.union(add_row_df)
df_final.filter(col("id") == '5180').show()
df_final.orderBy(desc('id')).show()

+----+----------+--------------------+-----------------+-------------+--------------------+--------------+------------+
|  id|      date|                 url|             isbn|citationscore|           publisher|         title|editor_array|
+----+----------+--------------------+-----------------+-------------+--------------------+--------------+------------+
|5180|2022-10-02|https://www.thebl...|978-8-817-08378-2|    0.5652352|Politecnico di Mi...|The black hole|[5052, 5180]|
+----+----------+--------------------+-----------------+-------------+--------------------+--------------+------------+

+----+----------+--------------------+-----------------+-------------+--------------------+--------------------+--------------------+
|  id|      date|                 url|             isbn|citationscore|           publisher|               title|        editor_array|
+----+----------+--------------------+-----------------+-------------+--------------------+--------------------+-------------------